# **Значения Шепли с причинным графом: asymmetric, causal Shapley и Shapley Flow**

Практика к модулю [«Каузальность и линейные модели»](https://ai-interpretability.school/learn/3/0/0), урок «Значения Шепли с причинным графом».

В уроке варианты значений Шепли сравнивались на двух признаках и одной модели. Здесь мы посчитаем их своим кодом на модели проката велосипедов, где известен граф признаков, и сверим маргинальный вариант с пакетом `shap`. Проверим четыре утверждения урока:

1. маргинальные (interventional) значения SHAP не дают признаку-причине ничего, если модель на него не смотрит, а условные (observational) дают, но не различают граф;
2. asymmetric Shapley values (ASV) отдают причине весь вклад, который проходит через ее следствия;
3. causal Shapley values меняются вместе с графом при тех же данных;
4. Shapley Flow раскладывает тот же прогноз по ребрам графа, и сумма по любому разрезу графа одна и та же.

План тетради:

1. данные и модель;
2. четыре функции ценности;
3. симметричные и асимметричные значения Шепли;
4. другой граф при тех же данных;
5. Shapley Flow;
6. где ломается: неустойчивость ASV;
7. сверка с `shap`.

Тетрадь идет на процессоре, прогон занимает несколько секунд.

In [ ]:
import itertools
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

SEED = 0
names = ['season', 'temperature', 'humidity']

## 1. Данные и модель

Прокат велосипедов зависит от температуры и влажности; сезон сам на прокат не действует, но задает температуру. Граф признаков — $X_1 \to X_2$, $X_3$ отдельно:

$$X_1 \sim \mathcal N(0,1),\quad X_2 = 0.8\,X_1 + \varepsilon_2,\ \varepsilon_2 \sim \mathcal N(0, 0.6^2),\quad X_3 \sim \mathcal N(0,1),\quad Y = 2X_2 - X_3 + \varepsilon_Y,$$

где $X_1$ — сезон (от зимы к лету), $X_2$ — температура, $X_3$ — влажность, все стандартизованы; $Y$ — прокат в условных единицах. Модель $f$ — линейная регрессия по трем признакам. Объясняем прогноз для одного дня: зима ($x_1 = -1$), но тепло для зимы ($x_2 = 0.5$), влажность чуть выше обычной ($x_3 = 0.3$).

In [ ]:
rng = np.random.default_rng(SEED)
n = 5000
X1 = rng.normal(size=n)
X2 = 0.8 * X1 + rng.normal(0, 0.6, n)
X3 = rng.normal(size=n)
Y = 2 * X2 - X3 + rng.normal(0, 0.5, n)
X = np.c_[X1, X2, X3]

model = LinearRegression().fit(X, Y)
f = lambda z: model.intercept_ + z @ model.coef_
x = np.array([-1.0, 0.5, 0.3])
mu, Sigma = X.mean(0), np.cov(X.T)
print('коэффициенты модели:', dict(zip(names, model.coef_.round(3).tolist())))
print(f'f(x) = {f(x):.3f} | E f = {f(mu):.3f} | f(x) - E f = {f(x) - f(mu):.3f}')

Модель почти не смотрит на сезон: коэффициент около нуля. Прогноз для этого дня на 0.705 выше среднего.

## 2. Четыре функции ценности

Значение Шепли признака $i$ — средний прирост функции ценности $v(S)$ при добавлении $i$ к признакам, которые уже известны. Варианты отличаются тем, что подставляют на место неизвестных признаков $\bar S$:

- **маргинальная** (interventional SHAP в пакете `shap`): $v(S) = E\big[f(x_S, X_{\bar S})\big]$ — неизвестные признаки из их общего распределения, независимо от известных;
- **условная** (observational): $v(S) = E\big[f(X)\mid X_S = x_S\big]$ — из распределения при известных $x_S$;
- **причинная** ([Heskes et al., 2020](https://arxiv.org/abs/2011.01625)): $v(S) = E\big[f(X)\mid do(X_S = x_S)\big]$ — из распределения после вмешательства по графу.

Модель линейная, поэтому $E[f(\cdot)] = f(E[\cdot])$, и каждую функцию ценности считаем через среднее неизвестных признаков. Условное среднее для гауссовых признаков — $\mu_{\bar S} + \Sigma_{\bar S S}\Sigma_{SS}^{-1}(x_S - \mu_S)$. Среднее после вмешательства считаем по уравнениям графа: признак из $S$ равен $x_i$, остальные — своему уравнению от уже посчитанных родителей. Уравнение $X_2$ от $X_1$ оцениваем регрессией.

In [ ]:
d = 3
eq2 = LinearRegression().fit(X[:, [0]], X[:, 1])        # X2 = c + alpha * X1 + шум
alpha, c2 = eq2.coef_[0], eq2.intercept_
print(f'alpha = {alpha:.3f}')


def v_marginal(S):
    z = mu.copy()
    z[S] = x[S]
    return f(z)


def v_conditional(S):
    z = x.copy()
    R = [j for j in range(d) if j not in S]
    z[R] = mu[R] if not S else mu[R] + Sigma[np.ix_(R, S)] @ np.linalg.solve(Sigma[np.ix_(S, S)], x[S] - mu[S])
    return f(z)


def v_causal_chain(S):
    """Граф X1 -> X2, X3 отдельно: признаки из S фиксированы вмешательством, остальные — по уравнениям."""
    z = np.empty(d)
    z[0] = x[0] if 0 in S else mu[0]
    z[1] = x[1] if 1 in S else c2 + alpha * z[0]
    z[2] = x[2] if 2 in S else mu[2]
    return f(z)


for name, v in (('маргинальная', v_marginal), ('условная', v_conditional), ('причинная', v_causal_chain)):
    print(f'{name:13s}', {str(S): round(float(v(S)), 3) for S in ([], [0], [1], [0, 1])})

Различие видно уже на коалициях. Знать один сезон, $S = \{1\}$: маргинальная функция почти не меняется, а условная и причинная падают — зимой ждут холода. Знать одну температуру, $S = \{2\}$: условная функция подтягивает сезон к теплу, а причинная нет — вмешательство в температуру сезон не меняет.

## 3. Симметричные и асимметричные значения Шепли

Значение Шепли усредняет прирост по порядкам добавления признаков $\pi$:

$$\phi_i = \sum_{\pi} w(\pi)\,\big[v(\{j : \pi(j) \le \pi(i)\}) - v(\{j : \pi(j) < \pi(i)\})\big].$$

Симметричный вариант берет все $3! = 6$ порядков с весом $1/6$. Асимметричный ([Frye, Rowat и Feige, 2020](https://arxiv.org/abs/1910.06358)) оставляет только порядки, где причина идет раньше следствия: сезон раньше температуры, — их три, с весом $1/3$.

**Задание 1. Напишите функцию `shapley(v, orders)`, которая усредняет приросты по списку порядков, и посчитайте симметричные значения Шепли с причинной функцией ценности `v_causal_chain`. В ответ — значение для сезона, округленное до трех знаков после точки.**

In [ ]:
# Ваш код здесь

Теперь все пять вариантов на одном прогнозе.

**Задание 2. Посчитайте асимметричные значения Шепли с условной функцией ценности (это ASV Frye et al.) на порядках, где сезон идет раньше температуры, и сведите в таблицу с остальными вариантами. В ответ — значение ASV для сезона, округленное до трех знаков после точки.**

In [ ]:
# Ваш код здесь

Сумма во всех строках одна — 0.705, разница между прогнозом и средним: эффективность держится у всех вариантов. Маргинальные значения дают сезону почти ноль и весь вклад отдают температуре. ASV отдает сезону весь эффект, который идет через температуру, $\hat\alpha\hat\beta_2(x_1 - \mu_1) \approx -1.6$: зимой проката меньше, потому что холодно, и это записывается на счет сезона, а температуре остается только то, насколько этот день теплее обычного для зимы. Симметричные варианты делят этот эффект между сезоном и температурой пополам.

На цепочке условные и причинные значения почти совпадают. Различаются они там, где связь признаков идет не от одного к другому.

## 4. Другой граф при тех же данных

Пусть сезон и температура связаны не напрямую, а через общую причину — например, через климат региона, который в данных не записан. Распределение признаков то же, но вмешательство в сезон теперь температуру не меняет:

$$v(S) = E\big[f(X)\mid do(X_S = x_S)\big],\qquad E[X_2 \mid do(X_1 = x_1)] = \mu_2.$$

In [ ]:
def v_causal_confounded(S):
    """Сезон и температура — следствия общей причины: вмешательство в один признак не меняет другой."""
    z = mu.copy()
    z[S] = x[S]
    return f(z)


compare = pd.DataFrame({
    'условные (observational SHAP)': shapley(v_conditional, all_orders),
    'ASV, если считать сезон причиной': shapley(v_conditional, causal_orders),
    'причинные, граф с общей причиной': shapley(v_causal_confounded, all_orders),
    'причинные, цепочка': shapley(v_causal_chain, all_orders),
}, index=names).T
compare.round(3)

Условные значения и ASV от графа не зависят: они видят только распределение данных, а оно одно. Причинные значения при графе с общей причиной совпадают с маргинальными — сезон модели не нужен и получает почти ноль (0.012). При цепочке сезон получает свою половину эффекта через температуру. Какой ответ верен, данные сказать не могут; это решает граф, и его выбор надо писать рядом с объяснением.

## 5. Shapley Flow

[Wang, Wiens и Lundberg, 2021](https://arxiv.org/abs/2010.14592) распределяют разность $f(x) - f(x')$ между ребрами графа, где $x'$ — фоновая точка. У каждой вершины, кроме корней, есть свой шум, и он тоже источник: $\varepsilon_2 = x_2 - c_2 - \hat\alpha x_1$. Граф для Shapley Flow: $X_1 \to X_2$, $\varepsilon_2 \to X_2$, и ребра $X_1 \to f$, $X_2 \to f$, $X_3 \to f$ в модель.

Значение Shapley Flow пути — средний прирост выхода, когда по этому пути проходит значение объекта $x$ вместо фонового, по всем порядкам обхода графа в глубину; ребро получает сумму значений путей, которые через него проходят. Когда и модель, и уравнения линейны, прирост пути не зависит от порядка и равен произведению коэффициентов вдоль пути на сдвиг источника $x_s - x'_s$. Фон — средняя точка $x' = \mu$, у нее $\varepsilon'_2 = 0$.

In [ ]:
b1, b2, b3 = model.coef_
eps2 = x[1] - c2 - alpha * x[0]
paths = {
    'X1 -> f': b1 * (x[0] - mu[0]),
    'X1 -> X2 -> f': alpha * b2 * (x[0] - mu[0]),
    'eps2 -> X2 -> f': b2 * eps2,
    'X3 -> f': b3 * (x[2] - mu[2]),
}
print({p: round(float(val), 3) for p, val in paths.items()})
print(f'сумма путей {sum(paths.values()):.3f} | f(x) - f(mu) {f(x) - f(mu):.3f}')

**Задание 3. Посчитайте значения Shapley Flow для ребер $X_1 \to X_2$ и $X_2 \to f$ как сумму значений путей, которые через них проходят. Проверьте, что сумма по разрезу у модели ($X_1 \to f$, $X_2 \to f$, $X_3 \to f$) и по разрезу у источников ($X_1 \to f$, $X_1 \to X_2$, $\varepsilon_2 \to X_2$, $X_3 \to f$) одна и та же. В ответ — значение ребра $\varepsilon_2 \to X_2$, округленное до трех знаков после точки.**

In [ ]:
# Ваш код здесь

Разрез у модели дает маргинальные значения SHAP: ребро $X_2 \to f$ получает $\hat\beta_2(x_2 - \mu_2)$. Разрез у источников раскладывает то же число глубже: из вклада температуры $-1.6$ пришло от сезона, а $+2.6$ — от того, что день теплее обычного для зимы. Сумма ребер, выходящих из $X_1$, равна $-1.595$, ребро $\varepsilon_2 \to X_2$ — $2.592$: это причинные асимметричные значения сезона и температуры из раздела 3. Авторы доказывают (раздел 3.4 статьи), что при графе, который описывает все зависимости признаков, значения Shapley Flow у источников совпадают с ASV; у нас ASV отличаются примерно на 0.013, потому что условное среднее посчитано по выборочной ковариации, а уравнение графа — отдельной регрессией.

## 6. Где ломается: неустойчивость ASV

[Heskes et al., 2020](https://arxiv.org/abs/2011.01625) показывают, что ASV зависят от порядка признаков, даже когда связи между ними нет. Модель — XOR двух независимых равновероятных бинарных признаков, объясняем прогноз $f(1, 0) = 1$. Добавим в граф ребро $X_1 \to X_2$ нулевой силы: распределение не меняется, меняется только список допустимых порядков.

In [ ]:
xor_point = np.array([1, 0])
grid = np.array(list(itertools.product([0, 1], repeat=2)))     # равновероятные пары значений


def v_xor(S):
    rows = grid[np.all(grid[:, S] == xor_point[S], axis=1)] if S else grid
    return np.mean(rows[:, 0] ^ rows[:, 1])


def shapley2(v, orders):
    phi = np.zeros(2)
    for order in orders:
        known = []
        for i in order:
            phi[i] += v(known + [i]) - v(known)
            known = known + [i]
    return phi / len(orders)


print('симметричные:', shapley2(v_xor, [(0, 1), (1, 0)]))
print('ASV, X1 раньше X2:', shapley2(v_xor, [(0, 1)]))

Ребро нулевой силы перевело весь вклад с $X_1$ на $X_2$: 0.25 и 0.25 стали 0 и 0.5. Симметричные значения, и условные, и причинные, от такого ребра не меняются. Поэтому асимметричный вариант берут, когда порядок причин действительно известен — например, задан временем, — а не для того, чтобы проверить догадку о графе.

## 7. Сверка с `shap`

Маргинальные значения считает `shap` с маской `Independent`: неизвестные признаки берутся из фоновой выборки независимо от известных. Фон — все 5000 объектов, тогда среднее фона совпадает с `mu`.

In [ ]:
import shap

explainer = shap.explainers.Exact(model.predict, shap.maskers.Independent(X, max_samples=len(X)))
phi_shap = explainer(x[None, :]).values[0]
print('shap, Independent:', phi_shap.round(3))
print('наши маргинальные:', shapley(v_marginal, all_orders).round(3))

ASV и causal Shapley values реализованы в R-пакете [`shapr`](https://github.com/NorskRegnesentral/shapr) (лицензия MIT): у функции `explain` есть аргументы `asymmetric`, `causal_ordering` и `confounding`; Python-обертке `pyshapr` нужен установленный R. Код Shapley Flow авторы опубликовали без лицензии. Поэтому здесь своя реализация, в которой видна каждая формула. Для нелинейной модели и нелинейных уравнений функции ценности считают выборкой: неизвестные признаки сэмплируют из условного распределения или из уравнений графа после вмешательства и усредняют $f$.

## Что унести из тетради

- **Функция ценности решает, что объясняется.** Маргинальная объясняет саму модель и отдает сезону почти ноль; условная и причинная объясняют прогноз вместе с миром, где сезон задает температуру.
- **ASV отдают причине все, что идет через следствия.** Сезон получает −1.6, температура — то, насколько день теплее обычного для зимы.
- **Причинные значения зависят от графа, условные — нет.** При тех же данных граф с общей причиной дает сезону ноль, цепочка — половину эффекта через температуру.
- **Shapley Flow раскладывает прогноз по ребрам.** Сумма по любому разрезу одна, разрез у модели дает маргинальные значения, разрез у источников — причинные асимметричные.
- **Асимметричный вариант неустойчив к ребру нулевой силы.** На XOR такое ребро переносит весь вклад с одного признака на другой.

Источники:
1. [Frye, Rowat и Feige, 2020. Asymmetric Shapley values: incorporating causal knowledge into model-agnostic explainability](https://arxiv.org/abs/1910.06358).
2. [Heskes, Sijben, Bucur и Claassen, 2020. Causal Shapley Values: Exploiting Causal Knowledge to Explain Individual Predictions of Complex Models](https://arxiv.org/abs/2011.01625).
3. [Wang, Wiens и Lundberg, 2021. Shapley Flow: A Graph-based Approach to Interpreting Model Predictions](https://arxiv.org/abs/2010.14592).
4. [Janzing, Minorics и Blöbaum, 2020. Feature relevance quantification in explainable AI: A causal problem](https://arxiv.org/abs/1910.13413).